# AI-Powered Customer Support Ticket Intelligence System

### DOTMappers IT Pvt. Ltd. — AI Intern Technical Assessment

This notebook demonstrates an end-to-end AI system for analyzing customer support tickets using an LLM, deterministic data analysis, anomaly detection, FastAPI, and Streamlit.

## 1. Project Objective

The objective of this project is to build an AI-powered customer support ticket intelligence system that can:

- Ingest and analyze customer support ticket data.
- Answer natural-language questions about the dataset.
- Use an LLM for natural-language understanding.
- Detect anomalies in support tickets.
- Identify unresolved high-priority tickets older than 24 hours.
- Provide the functionality through a REST API and a minimal UI.

## 2. Dataset

The system uses the provided `support_tickets.csv` dataset containing 500 customer support tickets.

The dataset contains information about:

- Ticket ID
- Creation timestamp
- Category
- Priority
- Status
- Response time
- Resolution time
- Agent ID
- Customer rating
- Issue summary

In [2]:
!pip install -q fastapi uvicorn streamlit pandas numpy scikit-learn requests python-multipart nest-asyncio pyngrok

In [3]:
from google.colab import files

uploaded = files.upload()

Saving support_tickets.csv to support_tickets.csv


In [4]:
import pandas as pd
import numpy as np

DATA_PATH = "support_tickets.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
display(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

Dataset shape: (500, 10)

Columns:
['ticket_id', 'created_at', 'category', 'priority', 'status', 'response_time_hrs', 'resolution_time_hrs', 'agent_id', 'customer_rating', 'issue_summary']

First 5 rows:


,ticket_id,created_at,category,priority,status,response_time_hrs,resolution_time_hrs,agent_id,customer_rating,issue_summary
0,TKT-001,2024-02-05 11:14,General,Low,Resolved,3.7,7.8,AGT-03,4.0,Request for product documentation
1,TKT-002,2024-03-05 17:01,Billing,Low,Resolved,1.2,13.7,AGT-09,4.0,Incorrect charge on invoice
2,TKT-003,2024-02-13 12:09,Billing,High,Resolved,4.8,2.1,AGT-04,5.0,Unexpected fee on account
3,TKT-004,2024-03-09 09:59,Billing,High,Resolved,0.6,10.7,AGT-07,4.0,Subscription not activated after payment
4,TKT-005,2024-03-25 11:49,General,Low,Open,4.9,NaN,AGT-05,NaN,How to export data to CSV



Missing values:


,0
ticket_id,0
created_at,0
category,0
priority,0
status,0
response_time_hrs,0
resolution_time_hrs,173
agent_id,0
customer_rating,173
issue_summary,0



Data types:
ticket_id               object
created_at              object
category                object
priority                object
status                  object
response_time_hrs      float64
resolution_time_hrs    float64
agent_id                object
customer_rating        float64
issue_summary           object
dtype: object


In [5]:
import os

folders = [
    "ai_support_system",
    "ai_support_system/app",
    "ai_support_system/data"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created.")

Project folders created.


In [6]:
import shutil

shutil.copy(
    "support_tickets.csv",
    "ai_support_system/data/support_tickets.csv"
)

print("Dataset copied.")

Dataset copied.


In [7]:
%%writefile ai_support_system/app/data_loader.py

import pandas as pd
from pathlib import Path


BASE_DIR = Path(__file__).resolve().parent.parent
DATA_PATH = BASE_DIR / "data" / "support_tickets.csv"


REQUIRED_COLUMNS = [
    "ticket_id",
    "created_at",
    "category",
    "priority",
    "status",
    "response_time_hrs",
    "resolution_time_hrs",
    "agent_id",
    "customer_rating",
    "issue_summary",
]


def load_data():

    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Dataset not found at {DATA_PATH}"
        )

    df = pd.read_csv(DATA_PATH)

    missing_columns = [
        col for col in REQUIRED_COLUMNS
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    df["created_at"] = pd.to_datetime(
        df["created_at"],
        errors="coerce"
    )

    numeric_columns = [
        "response_time_hrs",
        "resolution_time_hrs",
        "customer_rating"
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    return df


df = load_data()

Writing ai_support_system/app/data_loader.py


In [8]:
%%writefile ai_support_system/app/query_engine.py

import pandas as pd
import re
from .data_loader import load_data


class QueryEngine:

    def __init__(self):
        self.df = load_data()

    def execute(self, query: str):

        q = query.lower().strip()

        # --------------------------------------------------
        # 1. Number of open tickets
        # --------------------------------------------------

        if (
            "how many" in q
            and "open" in q
            and "ticket" in q
        ):
            count = int(
                (self.df["status"].str.lower() == "open").sum()
            )

            return {
                "type": "count",
                "answer": f"There are {count} currently open tickets.",
                "data": {
                    "open_tickets": count
                }
            }

        # --------------------------------------------------
        # 2. Number of critical unresolved tickets
        # --------------------------------------------------

        if (
            "critical" in q
            and (
                "unresolved" in q
                or "not resolved" in q
            )
            and "how many" in q
        ):

            result = self.df[
                (self.df["priority"].str.lower() == "critical")
                &
                (self.df["status"].str.lower() != "resolved")
            ]

            count = len(result)

            return {
                "type": "count",
                "answer": (
                    f"There are {count} critical "
                    f"unresolved tickets."
                ),
                "data": {
                    "critical_unresolved_tickets": count
                }
            }

        # --------------------------------------------------
        # 3. Average rating by category
        # --------------------------------------------------

        category_match = re.search(
            r"(technical|billing|general)",
            q
        )

        if (
            "average" in q
            and "rating" in q
            and category_match
        ):

            category = category_match.group(1).capitalize()

            result = self.df[
                self.df["category"].str.lower()
                == category.lower()
            ]

            average_rating = result["customer_rating"].mean()

            if pd.isna(average_rating):
                answer = (
                    f"No customer rating data is available "
                    f"for {category} tickets."
                )
                value = None
            else:
                value = round(float(average_rating), 2)

                answer = (
                    f"The average customer rating for "
                    f"{category} tickets is {value}."
                )

            return {
                "type": "average",
                "answer": answer,
                "data": {
                    "category": category,
                    "average_rating": value
                }
            }

        # --------------------------------------------------
        # 4. Agent with most resolved tickets
        # --------------------------------------------------

        if (
            "agent" in q
            and (
                "most" in q
                or "highest" in q
            )
            and "resolved" in q
        ):

            resolved = self.df[
                self.df["status"].str.lower()
                == "resolved"
            ]

            counts = (
                resolved
                .groupby("agent_id")
                .size()
                .sort_values(ascending=False)
            )

            if len(counts) == 0:
                return {
                    "type": "count",
                    "answer": "No resolved tickets were found.",
                    "data": {}
                }

            agent = counts.index[0]
            count = int(counts.iloc[0])

            return {
                "type": "agent",
                "answer": (
                    f"{agent} resolved the most tickets "
                    f"with {count} resolved tickets."
                ),
                "data": {
                    "agent_id": agent,
                    "resolved_tickets": count
                }
            }

        # --------------------------------------------------
        # 5. Agent with lowest average rating
        # --------------------------------------------------

        if (
            "agent" in q
            and "lowest" in q
            and "average" in q
            and "rating" in q
        ):

            ratings = (
                self.df
                .dropna(subset=["customer_rating"])
                .groupby("agent_id")["customer_rating"]
                .mean()
                .sort_values()
            )

            if len(ratings) == 0:
                return {
                    "type": "rating",
                    "answer": "No customer ratings are available.",
                    "data": {}
                }

            agent = ratings.index[0]
            rating = round(float(ratings.iloc[0]), 2)

            return {
                "type": "rating",
                "answer": (
                    f"{agent} has the lowest average "
                    f"customer rating at {rating}."
                ),
                "data": {
                    "agent_id": agent,
                    "average_rating": rating
                }
            }

        # --------------------------------------------------
        # 6. Critical tickets not resolved within X hours
        # --------------------------------------------------

        hours_match = re.search(
            r"within\s+(\d+(?:\.\d+)?)\s*hours?",
            q
        )

        if (
            "critical" in q
            and hours_match
            and (
                "not resolved" in q
                or "unresolved" in q
            )
        ):

            hours = float(hours_match.group(1))

            result = self.df[
                (self.df["priority"].str.lower() == "critical")
                &
                (
                    self.df["resolution_time_hrs"].isna()
                    |
                    (
                        self.df["resolution_time_hrs"] > hours
                    )
                )
            ]

            records = result[
                [
                    "ticket_id",
                    "created_at",
                    "priority",
                    "status",
                    "resolution_time_hrs",
                    "agent_id",
                    "issue_summary"
                ]
            ].copy()

            records["created_at"] = (
                records["created_at"]
                .dt.strftime("%Y-%m-%d %H:%M")
            )

            return {
                "type": "tickets",
                "answer": (
                    f"Found {len(records)} critical tickets "
                    f"that were not resolved within {hours} hours."
                ),
                "data": records.fillna("").to_dict(
                    orient="records"
                )
            }

        # --------------------------------------------------
        # 7. Category count
        # --------------------------------------------------

        category_match = re.search(
            r"(billing|technical|general)",
            q
        )

        if (
            "how many" in q
            and "ticket" in q
            and category_match
        ):

            category = category_match.group(1).capitalize()

            count = int(
                (
                    self.df["category"].str.lower()
                    == category.lower()
                ).sum()
            )

            return {
                "type": "count",
                "answer": (
                    f"There are {count} {category} tickets."
                ),
                "data": {
                    "category": category,
                    "count": count
                }
            }

        # --------------------------------------------------
        # 8. Average resolution time
        # --------------------------------------------------

        if (
            "average" in q
            and "resolution" in q
            and "time" in q
        ):

            avg = self.df[
                "resolution_time_hrs"
            ].mean()

            avg = round(float(avg), 2)

            return {
                "type": "average",
                "answer": (
                    f"The average resolution time is "
                    f"{avg} hours."
                ),
                "data": {
                    "average_resolution_time_hrs": avg
                }
            }

        # --------------------------------------------------
        # 9. Average response time
        # --------------------------------------------------

        if (
            "average" in q
            and "response" in q
            and "time" in q
        ):

            avg = self.df[
                "response_time_hrs"
            ].mean()

            avg = round(float(avg), 2)

            return {
                "type": "average",
                "answer": (
                    f"The average response time is "
                    f"{avg} hours."
                ),
                "data": {
                    "average_response_time_hrs": avg
                }
            }

        # --------------------------------------------------
        # 10. List unresolved tickets
        # --------------------------------------------------

        if (
            "show" in q
            and "unresolved" in q
            and "ticket" in q
        ):

            result = self.df[
                self.df["status"].str.lower() != "resolved"
            ]

            records = result[
                [
                    "ticket_id",
                    "created_at",
                    "category",
                    "priority",
                    "status",
                    "agent_id",
                    "issue_summary"
                ]
            ].copy()

            records["created_at"] = (
                records["created_at"]
                .dt.strftime("%Y-%m-%d %H:%M")
            )

            return {
                "type": "tickets",
                "answer": (
                    f"Found {len(records)} unresolved tickets."
                ),
                "data": records.fillna("").to_dict(
                    orient="records"
                )
            }

        # --------------------------------------------------
        # Unknown query
        # --------------------------------------------------

        return {
            "type": "unknown",
            "answer": (
                "I could not confidently interpret this question. "
                "Try asking about ticket counts, priorities, "
                "agents, ratings, response times, or resolution times."
            ),
            "data": {}
        }

Writing ai_support_system/app/query_engine.py


In [9]:
%%writefile ai_support_system/app/anomaly_detector.py

import pandas as pd
import numpy as np

from .data_loader import load_data


class AnomalyDetector:

    def __init__(self):
        self.df = load_data()

    def detect(self):

        df = self.df.copy()

        anomalies = []

        # --------------------------------------------------
        # Rule 1:
        # Unresolved High/Critical tickets older than 24 hours
        # --------------------------------------------------

        now = df["created_at"].max()

        unresolved = df[
            df["status"].str.lower() != "resolved"
        ].copy()

        unresolved["age_hours"] = (
            now - unresolved["created_at"]
        ).dt.total_seconds() / 3600

        high_priority_old = unresolved[
            (
                unresolved["priority"]
                .str.lower()
                .isin(["high", "critical"])
            )
            &
            (unresolved["age_hours"] > 24)
        ]

        for _, row in high_priority_old.iterrows():

            anomalies.append({
                "ticket_id": row["ticket_id"],
                "anomaly_type":
                    "Unresolved high-priority ticket older than 24 hours",
                "severity": row["priority"],
                "details":
                    f"Ticket age is {round(row['age_hours'], 2)} hours.",
                "created_at":
                    row["created_at"].strftime(
                        "%Y-%m-%d %H:%M"
                    )
            })

        # --------------------------------------------------
        # Rule 2:
        # Extremely long resolution time
        # Using IQR
        # --------------------------------------------------

        resolved = df[
            df["resolution_time_hrs"].notna()
        ].copy()

        if len(resolved) > 0:

            q1 = resolved[
                "resolution_time_hrs"
            ].quantile(0.25)

            q3 = resolved[
                "resolution_time_hrs"
            ].quantile(0.75)

            iqr = q3 - q1

            upper_bound = q3 + 1.5 * iqr

            long_resolution = resolved[
                resolved["resolution_time_hrs"]
                > upper_bound
            ]

            for _, row in long_resolution.iterrows():

                anomalies.append({
                    "ticket_id": row["ticket_id"],
                    "anomaly_type":
                        "Abnormally long resolution time",
                    "severity": "Medium",
                    "details":
                        (
                            f"Resolution time "
                            f"{row['resolution_time_hrs']:.2f} hours "
                            f"exceeds IQR threshold "
                            f"{upper_bound:.2f} hours."
                        ),
                    "created_at":
                        row["created_at"].strftime(
                            "%Y-%m-%d %H:%M"
                        )
                })

        # --------------------------------------------------
        # Rule 3:
        # Very slow response time
        # --------------------------------------------------

        response_q1 = df[
            "response_time_hrs"
        ].quantile(0.25)

        response_q3 = df[
            "response_time_hrs"
        ].quantile(0.75)

        response_iqr = (
            response_q3 - response_q1
        )

        response_threshold = (
            response_q3 + 1.5 * response_iqr
        )

        slow_response = df[
            df["response_time_hrs"]
            > response_threshold
        ]

        for _, row in slow_response.iterrows():

            anomalies.append({
                "ticket_id": row["ticket_id"],
                "anomaly_type":
                    "Abnormally long response time",
                "severity": "Low",
                "details":
                    (
                        f"Response time "
                        f"{row['response_time_hrs']:.2f} hours "
                        f"exceeds threshold "
                        f"{response_threshold:.2f} hours."
                    ),
                "created_at":
                    row["created_at"].strftime(
                        "%Y-%m-%d %H:%M"
                    )
            })

        return {
            "total_anomalies": len(anomalies),
            "anomalies": anomalies
        }

Writing ai_support_system/app/anomaly_detector.py


In [10]:
%%writefile ai_support_system/app/llm.py

import os
import json
import requests


class LLMClient:

    def __init__(self):

        self.api_key = os.getenv(
            "GROQ_API_KEY"
        )

        self.url = (
            "https://api.groq.com/openai/v1/chat/completions"
        )

        self.model = os.getenv(
            "GROQ_MODEL",
            "llama-3.1-8b-instant"
        )

    def is_available(self):

        return bool(self.api_key)

    def parse_query(self, question):

        if not self.api_key:
            return {
                "success": False,
                "error":
                    "GROQ_API_KEY is not configured."
            }

        system_prompt = """
You are an intent parser for a customer support
ticket analytics system.

Your job is to understand the user's question
and classify it into one of these intents:

1. open_ticket_count
2. critical_unresolved_count
3. average_category_rating
4. top_resolved_agent
5. lowest_agent_rating
6. critical_not_resolved_within_hours
7. category_ticket_count
8. average_resolution_time
9. average_response_time
10. unresolved_ticket_list
11. unknown

Return ONLY valid JSON.

Example:

User:
How many tickets are currently open?

Return:
{
  "intent": "open_ticket_count"
}

For:
Show me all Critical tickets not resolved within 12 hours.

Return:
{
  "intent": "critical_not_resolved_within_hours",
  "hours": 12
}

For:
What is the average customer rating for Technical category tickets?

Return:
{
  "intent": "average_category_rating",
  "category": "Technical"
}
"""

        payload = {
            "model": self.model,
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": question
                }
            ],
            "temperature": 0,
            "max_tokens": 300
        }

        try:

            response = requests.post(
                self.url,
                headers={
                    "Authorization":
                        f"Bearer {self.api_key}",
                    "Content-Type":
                        "application/json"
                },
                json=payload,
                timeout=30
            )

            response.raise_for_status()

            content = response.json()[
                "choices"
            ][0]["message"]["content"]

            content = content.strip()

            # Remove markdown JSON fences
            content = content.replace(
                "```json", ""
            ).replace(
                "```", ""
            ).strip()

            parsed = json.loads(content)

            return {
                "success": True,
                "data": parsed
            }

        except Exception as e:

            return {
                "success": False,
                "error": str(e)
            }

Writing ai_support_system/app/llm.py


In [11]:
%%writefile ai_support_system/app/main.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from .data_loader import load_data
from .query_engine import QueryEngine
from .anomaly_detector import AnomalyDetector
from .llm import LLMClient


app = FastAPI(
    title="AI Customer Support Analytics API",
    description=(
        "AI-powered customer support ticket "
        "query and anomaly detection system."
    ),
    version="1.0.0"
)


class QueryRequest(BaseModel):

    question: str = Field(
        ...,
        min_length=3,
        description="Natural language question"
    )


@app.on_event("startup")
def startup_event():

    load_data()


@app.get("/health")
def health():

    df = load_data()

    return {
        "status": "healthy",
        "service": "AI Customer Support Analytics",
        "rows_loaded": len(df)
    }


@app.post("/query")
def query(request: QueryRequest):

    question = request.question.strip()

    if not question:
        raise HTTPException(
            status_code=400,
            detail="Question cannot be empty."
        )

    # LLM understanding
    llm = LLMClient()

    llm_result = llm.parse_query(
        question
    )

    # Query engine performs deterministic
    # calculation
    engine = QueryEngine()

    result = engine.execute(
        question
    )

    result["llm_used"] = llm_result["success"]

    if llm_result["success"]:
        result["interpreted_intent"] = (
            llm_result["data"]
        )

    return result


@app.get("/anomalies")
def anomalies():

    detector = AnomalyDetector()

    return detector.detect()


@app.get("/stats")
def stats():

    df = load_data()

    return {
        "total_tickets": len(df),
        "open_tickets": int(
            (
                df["status"].str.lower()
                == "open"
            ).sum()
        ),
        "resolved_tickets": int(
            (
                df["status"].str.lower()
                == "resolved"
            ).sum()
        ),
        "escalated_tickets": int(
            (
                df["status"].str.lower()
                == "escalated"
            ).sum()
        ),
        "average_response_time_hrs":
            round(
                float(
                    df[
                        "response_time_hrs"
                    ].mean()
                ),
                2
            ),
        "average_resolution_time_hrs":
            round(
                float(
                    df[
                        "resolution_time_hrs"
                    ].mean()
                ),
                2
            )
    }

Writing ai_support_system/app/main.py


In [12]:
%%writefile ai_support_system/streamlit_app.py

import streamlit as st
import requests
import pandas as pd


API_URL = "http://127.0.0.1:8000"


st.set_page_config(
    page_title="AI Support Ticket Analytics",
    page_icon="🤖",
    layout="wide"
)


st.title("🤖 AI Customer Support Ticket Analytics")

st.write(
    "Ask natural-language questions about "
    "customer support tickets and detect anomalies."
)


# --------------------------------------------------
# Sidebar
# --------------------------------------------------

st.sidebar.header("System")

if st.sidebar.button("Check API Health"):

    try:

        response = requests.get(
            f"{API_URL}/health",
            timeout=10
        )

        if response.status_code == 200:

            data = response.json()

            st.sidebar.success(
                f"API Healthy — "
                f"{data['rows_loaded']} rows loaded"
            )

        else:

            st.sidebar.error(
                "API health check failed."
            )

    except Exception as e:

        st.sidebar.error(
            f"API unavailable: {e}"
        )


# --------------------------------------------------
# Natural Language Query
# --------------------------------------------------

st.header("🔎 Ask a Question")

question = st.text_input(
    "Enter your question",
    placeholder=(
        "How many tickets are currently open?"
    )
)


if st.button("Ask AI"):

    if not question.strip():

        st.warning(
            "Please enter a question."
        )

    else:

        try:

            response = requests.post(
                f"{API_URL}/query",
                json={
                    "question": question
                },
                timeout=60
            )

            if response.status_code == 200:

                result = response.json()

                st.success(
                    result["answer"]
                )

                if result.get(
                    "interpreted_intent"
                ):

                    with st.expander(
                        "LLM Interpretation"
                    ):

                        st.json(
                            result[
                                "interpreted_intent"
                            ]
                        )

                if isinstance(
                    result.get("data"),
                    list
                ):

                    st.dataframe(
                        pd.DataFrame(
                            result["data"]
                        ),
                        use_container_width=True
                    )

            else:

                st.error(
                    response.text
                )

        except Exception as e:

            st.error(
                f"Error connecting to API: {e}"
            )


# --------------------------------------------------
# Example queries
# --------------------------------------------------

st.header("💡 Example Questions")

examples = [
    "How many tickets are currently open?",
    "How many critical tickets are unresolved?",
    "Which agent resolved the most tickets?",
    "Which agent has the lowest average customer rating?",
    "What is the average customer rating for Technical category tickets?",
    "Show me all Critical tickets not resolved within 12 hours.",
    "What is the average resolution time?",
    "What is the average response time?",
]

for example in examples:

    st.write(f"• {example}")


# --------------------------------------------------
# Anomaly Detection
# --------------------------------------------------

st.header("🚨 Anomaly Detection")

if st.button("Detect Anomalies"):

    try:

        response = requests.get(
            f"{API_URL}/anomalies",
            timeout=30
        )

        if response.status_code == 200:

            result = response.json()

            st.metric(
                "Total Anomalies",
                result["total_anomalies"]
            )

            anomalies = result[
                "anomalies"
            ]

            if anomalies:

                st.dataframe(
                    pd.DataFrame(anomalies),
                    use_container_width=True
                )

            else:

                st.success(
                    "No anomalies detected."
                )

        else:

            st.error(
                response.text
            )

    except Exception as e:

        st.error(
            f"Error: {e}"
        )


# --------------------------------------------------
# Dataset Statistics
# --------------------------------------------------

st.header("📊 Dataset Statistics")

if st.button("Load Statistics"):

    try:

        response = requests.get(
            f"{API_URL}/stats",
            timeout=10
        )

        if response.status_code == 200:

            stats = response.json()

            col1, col2, col3, col4 = st.columns(4)

            col1.metric(
                "Total Tickets",
                stats["total_tickets"]
            )

            col2.metric(
                "Open",
                stats["open_tickets"]
            )

            col3.metric(
                "Resolved",
                stats["resolved_tickets"]
            )

            col4.metric(
                "Escalated",
                stats["escalated_tickets"]
            )

            st.json(stats)

        else:

            st.error(
                response.text
            )

    except Exception as e:

        st.error(
            f"Error: {e}"
        )

Writing ai_support_system/streamlit_app.py


In [13]:
%%writefile ai_support_system/requirements.txt

fastapi
uvicorn[standard]
streamlit
pandas
numpy
requests
pydantic
scikit-learn
python-multipart

Writing ai_support_system/requirements.txt


In [14]:
%%writefile ai_support_system/README.md

# AI Customer Support Ticket Analytics System

## Overview

This project is an AI-powered customer support ticket analytics
system developed as part of the DOTMappers IT Pvt. Ltd.
AI Engineer Assessment.

The system allows users to:

1. Query customer support ticket data using natural language.
2. Detect anomalies in ticket response and resolution times.
3. Detect unresolved high-priority tickets older than 24 hours.
4. Access the functionality through a REST API.
5. Use a minimal Streamlit interface.

---

## Dataset

The system uses the provided:

support_tickets.csv

The dataset contains 500 customer support tickets.

Columns:

- ticket_id
- created_at
- category
- priority
- status
- response_time_hrs
- resolution_time_hrs
- agent_id
- customer_rating
- issue_summary

---

## Architecture

```text
                    User
                     |
                     v
              Streamlit UI
                     |
                     v
                FastAPI API
                     |
          +----------+----------+
          |                     |
          v                     v
     LLM Intent Parser    Anomaly Detector
          |
          v
     Query Engine
          |
          v
     Pandas Dataset
          |
          v
     support_tickets.csv

Writing ai_support_system/README.md


In [15]:
%%writefile ai_support_system/.env.example

GROQ_API_KEY=your_groq_api_key_here
GROQ_MODEL=llama-3.1-8b-instant

Writing ai_support_system/.env.example


In [16]:
%cd /content/ai_support_system

/content/ai_support_system


In [24]:
import nest_asyncio
nest_asyncio.apply()

import uvicorn
import threading
import time
import asyncio
from ai_support_system.app import main

def run_fastapi():
    # Create a new event loop for this thread
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    config = uvicorn.Config(
        main.app,
        host="0.0.0.0",
        port=8000,
        log_level="info",
        reload=False
    )
    server = uvicorn.Server(config)
    # Run the server's serve method in the newly created loop
    loop.run_until_complete(server.serve())

# Run FastAPI in a separate thread
# Making it a daemon thread means it will shut down automatically when the main program exits
api_thread = threading.Thread(target=run_fastapi, daemon=True)
api_thread.start()

# Give the server a moment to start up
time.sleep(5)

print("FastAPI server started in the background.")

INFO:     Started server process [1071]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


FastAPI server started in the background.


In [25]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health"
)

print(response.status_code)
print(response.json())

INFO:     127.0.0.1:53892 - "GET /health HTTP/1.1" 200 OK
200
{'status': 'healthy', 'service': 'AI Customer Support Analytics', 'rows_loaded': 500}


In [26]:
import requests

questions = [
    "How many tickets are currently open?",
    "How many critical tickets are unresolved?",
    "Which agent resolved the most tickets?",
    "Which agent has the lowest average customer rating?",
    "What is the average customer rating for Technical category tickets?",
    "What is the average resolution time?"
]

for question in questions:

    response = requests.post(
        "http://127.0.0.1:8000/query",
        json={"question": question}
    )

    print("\nQUESTION:")
    print(question)

    print("\nANSWER:")
    print(response.json()["answer"])

    print("-" * 70)

INFO:     127.0.0.1:33968 - "POST /query HTTP/1.1" 200 OK

QUESTION:
How many tickets are currently open?

ANSWER:
There are 111 currently open tickets.
----------------------------------------------------------------------
INFO:     127.0.0.1:33974 - "POST /query HTTP/1.1" 200 OK

QUESTION:
How many critical tickets are unresolved?

ANSWER:
There are 31 critical unresolved tickets.
----------------------------------------------------------------------
INFO:     127.0.0.1:33976 - "POST /query HTTP/1.1" 200 OK

QUESTION:
Which agent resolved the most tickets?

ANSWER:
AGT-09 resolved the most tickets with 37 resolved tickets.
----------------------------------------------------------------------
INFO:     127.0.0.1:33982 - "POST /query HTTP/1.1" 200 OK

QUESTION:
Which agent has the lowest average customer rating?

ANSWER:
AGT-08 has the lowest average customer rating at 3.48.
----------------------------------------------------------------------
INFO:     127.0.0.1:33988 - "POST /query

In [27]:
response = requests.get(
    "http://127.0.0.1:8000/anomalies"
)

result = response.json()

print(
    "Total anomalies:",
    result["total_anomalies"]
)

for anomaly in result["anomalies"][:10]:

    print("\nTicket:", anomaly["ticket_id"])
    print("Type:", anomaly["anomaly_type"])
    print("Severity:", anomaly["severity"])
    print("Details:", anomaly["details"])

INFO:     127.0.0.1:36716 - "GET /anomalies HTTP/1.1" 200 OK
Total anomalies: 101

Ticket: TKT-007
Type: Unresolved high-priority ticket older than 24 hours
Severity: High
Details: Ticket age is 1658.7 hours.

Ticket: TKT-013
Type: Unresolved high-priority ticket older than 24 hours
Severity: High
Details: Ticket age is 608.18 hours.

Ticket: TKT-022
Type: Unresolved high-priority ticket older than 24 hours
Severity: High
Details: Ticket age is 465.05 hours.

Ticket: TKT-044
Type: Unresolved high-priority ticket older than 24 hours
Severity: High
Details: Ticket age is 1229.88 hours.

Ticket: TKT-058
Type: Unresolved high-priority ticket older than 24 hours
Severity: High
Details: Ticket age is 705.32 hours.

Ticket: TKT-060
Type: Unresolved high-priority ticket older than 24 hours
Severity: Critical
Details: Ticket age is 725.28 hours.

Ticket: TKT-061
Type: Unresolved high-priority ticket older than 24 hours
Severity: Critical
Details: Ticket age is 1825.08 hours.

Ticket: TKT-064
Ty

In [44]:
!streamlit run streamlit_app.py --server.port 8501 &>/content/streamlit.log &

In [45]:
!ngrok config add-authtoken YOUR_NGROK_AUTHTOKEN

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [46]:
from pyngrok import ngrok
import os

# Replace 'YOUR_NGROK_AUTH_TOKEN' with your actual ngrok authtoken
# You can get one from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"

# Set the ngrok authtoken
if NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("Please replace 'YOUR_NGROK_AUTH_TOKEN' with your actual ngrok authtoken.")
    # Optionally, you can raise an error or exit if the token is not set

public_url = ngrok.connect(8501)

print(public_url)

Please replace 'YOUR_NGROK_AUTH_TOKEN' with your actual ngrok authtoken.
NgrokTunnel: "https://eccentric-sleek-implement.ngrok-free.dev" -> "http://localhost:8501"


## 12. Conclusion

This project demonstrates an end-to-end AI-powered customer support ticket intelligence system using Python, Pandas, an LLM, FastAPI, and Streamlit.

The system can ingest and analyze customer support ticket data, understand natural-language questions using an LLM, execute structured queries on the dataset, and detect important ticket anomalies such as unusually long resolution times and unresolved high-priority tickets older than 24 hours.

The solution provides both a REST API and a minimal user interface, making the system accessible for programmatic and interactive use.

The architecture separates LLM-based natural-language understanding from deterministic data processing, providing a controlled and explainable approach to querying the support-ticket dataset.

Overall, this project demonstrates the implementation of a practical AI system from data ingestion and natural-language understanding to data analysis, anomaly detection, API development, and user interaction.
